# Uygulama 5 — Kara Veba'nın İzinde: pPCP1 Plazmiti

**Hafta 3** — GenBank formatı, SeqRecord, SeqIO (read/write/to_dict), özellikler (features), konum gösterimi

**Veri:** NC_005816.1 (GenBank)

---

**Senaryo:** Bir halk sağlığı laboratuvarı, veba etkeni *Yersinia pestis*'in virülansında rol oynayan pPCP1 plazmitini inceliyor. Plazmitteki **pla** geni (plazminojen aktivatörü) bakterinin dokulara yayılmasını sağlar.

Senden GenBank kaydını okuyup genleri tablolaştırmanı, pla proteinini doğrulamanı ve tüm proteinleri FASTA'ya yazmanı istiyorlar.

> **GenBank vs FASTA:** FASTA sadece dizi + başlık içerir. GenBank ise bir "zengin kayıt" — dizi + genler + proteinler + kaynaklar + her şey. Düşün: FASTA = düz metin, GenBank = Word belgesi (biçimlendirme, tablolar, dipnotlar dahil).

## Kalem-Kağıt Isınması

GenBank kaydında pla geni `CDS 6664..7602` olarak verilmiş.

**(a) Biopython'daki 0-tabanlı, yarı açık gösterim:**
- `[6663:7602]` (başlangıç -1, bitiş aynı)

**(b) Gen kaç bp?**
- 7602 − 6664 + 1 = **939 bp**

**(c) Dur kodonu dahil kaç kodon, protein kaç aa?**
- 939 / 3 = **313 kodon** (dur dahil)
- Protein = 313 − 1 = **312 amino asit**

**(d) `complement(4815..5888)` hangi iplik?**
- **Ters iplik** (complement = tamamlayıcı, yani − ipliği)

---
## Görev 0 — Kütüphaneler

In [ ]:
from Bio import Entrez, SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.SeqUtils import gc_fraction
from collections import Counter
import os

Entrez.email = "semra@example.com"
print("Hazır!")

---
## Görev 1 — GenBank Kaydını İndir ve Oku

Bu sefer FASTA değil, **GenBank** formatında indiriyoruz — çünkü gen anotasyonları (features) lazım.

`SeqIO.read()` GenBank dosyasını okuyunca bir `SeqRecord` nesnesi döner:
- `.id` → erişim numarası
- `.seq` → dizi
- `.annotations` → organizma, tarih, topoloji gibi meta veriler
- `.features` → genler, CDS'ler, promotörler vs.

In [ ]:
if os.path.exists("NC_005816.gb"):
    record = SeqIO.read("NC_005816.gb", "genbank")
    print("Yerel dosyadan okundu.")
else:
    # rettype="gbwithparts" → CONTIG kayıtlarında diziyi de getirir
    handle = Entrez.efetch(db="nucleotide", id="NC_005816.1",
                           rettype="gbwithparts", retmode="text")
    gb_text = handle.read()
    handle.close()
    with open("NC_005816.gb", "w") as f:
        f.write(gb_text)
    record = SeqIO.read("NC_005816.gb", "genbank")
    print("NCBI'dan indirildi ve kaydedildi.")

print(f"\nKimlik:    {record.id}")
print(f"Uzunluk:   {len(record.seq)} bp")
print(f"Organizma: {record.annotations.get('organism', '?')}")
print(f"Topoloji:  {record.annotations.get('topology', '?')}")
print(f"Tarih:     {record.annotations.get('date', '?')}")
print(f"GC oranı:  {gc_fraction(record.seq) * 100:.1f}%")

---
## Görev 2 — Özellik (Feature) Türlerini Say ve CDS Tablosu

GenBank kaydındaki **features** listesi, DNA üzerindeki biyolojik öğeleri tanımlar:
- `source` → kaynak organizma bilgisi
- `gene` → gen bölgesi
- `CDS` → kodlayan dizi (protein üreten bölge)
- `misc_feature` → diğer özellikler

Her CDS'in içinde:
- `.location` → başlangıç, bitiş, iplik
- `.qualifiers` → gen adı, ürün, çeviri, locus_tag vs.

In [ ]:
# Feature türlerini say
tur_sayilari = Counter(f.type for f in record.features)
print("Özellik türleri:")
for tur, sayi in tur_sayilari.most_common():
    print(f"  {tur:20s} {sayi}")

print(f"\nToplam CDS: {tur_sayilari.get('CDS', 0)}")

In [ ]:
# CDS tablosu
print(f"{'Locus Tag':<12} {'Gen':<8} {'İplik':>5} {'Başlangıç':>10} {'Bitiş':>8} {'Uzunluk':>8}  Ürün")
print("-" * 85)

cds_listesi = []
for f in record.features:
    if f.type == "CDS":
        locus = f.qualifiers.get("locus_tag", ["?"])[0]
        gen = f.qualifiers.get("gene", ["?"])[0]
        iplik = "+" if f.location.strand == 1 else "-"
        bas = int(f.location.start) + 1  # 1-tabanlı
        son = int(f.location.end)
        uzunluk = son - bas + 1
        urun = f.qualifiers.get("product", ["?"])[0]
        
        print(f"{locus:<12} {gen:<8} {iplik:>5} {bas:>10} {son:>8} {uzunluk:>8}  {urun}")
        cds_listesi.append(f)

---
## Görev 3 — pla Genini Çevir ve Doğrula

- `feature.extract(record.seq)` → feature'ın kapsadığı diziyi çıkarır (ipliği de dikkate alır!)
- Bakteriyel genetik kod = **tablo 11**
- `cds=True` → başlangıç kodonu kontrolü + son dur kodonu kontrolü
- GenBank kaydında `/translation` alanında zaten çeviri var — kendi çevirimizle karşılaştıracağız

In [ ]:
# pla genini bul
pla_feature = None
for f in record.features:
    if f.type == "CDS" and "pla" in f.qualifiers.get("gene", [""]):
        pla_feature = f
        break

if pla_feature:
    # Diziyi çıkar
    pla_dna = pla_feature.extract(record.seq)
    print(f"pla DNA uzunluğu: {len(pla_dna)} bp")
    
    # Çevir (tablo 11 = bakteriyel kod)
    pla_protein = pla_dna.translate(table=11, cds=True)
    print(f"pla protein uzunluğu: {len(pla_protein)} aa")
    print(f"İlk 30 aa: {pla_protein[:30]}")
    
    # Kayıttaki çeviriyle karşılaştır
    kayit_ceviri = pla_feature.qualifiers.get("translation", [""])[0]
    eslesme = str(pla_protein) == kayit_ceviri
    print(f"\nKayıttaki çeviriyle eşleşiyor mu? → {eslesme}")
    if not eslesme:
        # Farkları göster
        for i in range(min(len(pla_protein), len(kayit_ceviri))):
            if pla_protein[i] != kayit_ceviri[i]:
                print(f"  Fark pozisyon {i+1}: bizim={pla_protein[i]}, kayıt={kayit_ceviri[i]}")
else:
    print("pla geni bulunamadı!")

---
## Görev 4 — Tüm CDS'leri Çevir ve Karşılaştır

Tüm CDS'leri çevirip kayıttaki çevirilerle karşılaştıralım. Ayrıca başlangıç kodonlarına bakalım.

> **Neden GTG/TTG ile başlayan genler M ile başlıyor?**
> Bakterilerde ATG dışında GTG ve TTG de başlangıç kodonu olabilir. Ribozom bunları tanır ve yine **formil-metionin (fMet)** koyar. Bu yüzden protein hep M ile başlar.

In [ ]:
baslangic_kodonlari = []

for f in record.features:
    if f.type != "CDS":
        continue
    
    gen = f.qualifiers.get("gene", ["?"])[0]
    dna = f.extract(record.seq)
    baslangic = str(dna[:3])
    baslangic_kodonlari.append(baslangic)
    
    # Çevir
    try:
        protein = dna.translate(table=11, cds=True)
        kayit = f.qualifiers.get("translation", [""])[0]
        esit = str(protein) == kayit
        durum = "OK" if esit else "FARKLI"
    except Exception as e:
        durum = f"HATA: {e}"
    
    print(f"{gen:<8} başlangıç={baslangic}  {durum}")

print(f"\nBaşlangıç kodonları: {Counter(baslangic_kodonlari)}")
print(f"\nGTG/TTG ile başlayan genler yine M ile başlıyor çünkü:")
print(f"→ Ribozom alternatif başlangıç kodonlarını da tanır")
print(f"→ Her durumda formil-metionin (fMet) yerleştirir")
print(f"→ cds=True bunu otomatik halleder")

---
## Görev 5 — Tüm Proteinleri FASTA'ya Yaz

Her CDS'ten çevirilen proteini bir FASTA dosyasına yazacağız. Sonra `SeqIO.to_dict` ile geri okuyup pla'ya kimliğiyle erişeceğiz.

`SeqIO.to_dict` → FASTA dosyasını sözlüğe çevirir (anahtar = kayıt ID'si)

In [ ]:
# Protein kayıtları oluştur
protein_kayitlari = []

for f in record.features:
    if f.type != "CDS":
        continue
    
    locus = f.qualifiers.get("locus_tag", ["unknown"])[0]
    gen = f.qualifiers.get("gene", ["?"])[0]
    urun = f.qualifiers.get("product", ["?"])[0]
    
    dna = f.extract(record.seq)
    try:
        protein = dna.translate(table=11, cds=True)
    except:
        protein = dna.translate(table=11, to_stop=True)
    
    prot_record = SeqRecord(
        protein,
        id=locus,
        description=f"{gen} | {urun}"
    )
    protein_kayitlari.append(prot_record)

# FASTA'ya yaz
dosya = "pPCP1_proteinler.fasta"
SeqIO.write(protein_kayitlari, dosya, "fasta")
print(f"{len(protein_kayitlari)} protein '{dosya}' dosyasına yazıldı.")

In [ ]:
# Geri oku ve pla'ya eriş
protein_dict = SeqIO.to_dict(SeqIO.parse(dosya, "fasta"))

print(f"Sözlükteki anahtarlar: {list(protein_dict.keys())}")

# pla'nın locus_tag'ini bul
pla_locus = pla_feature.qualifiers.get("locus_tag", [None])[0]
if pla_locus and pla_locus in protein_dict:
    pla_rec = protein_dict[pla_locus]
    print(f"\npla proteinine erişim ({pla_locus}):")
    print(f"  Açıklama: {pla_rec.description}")
    print(f"  Uzunluk:  {len(pla_rec.seq)} aa")
    print(f"  İlk 30:   {pla_rec.seq[:30]}")

---
## Görev 6 — Plazmitin Doğrusal Haritası

Genleri oklarla gösterip, iplik yönüne göre yukarı (ileri) veya aşağı (ters) çizeceğiz.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 3))

# Ana çizgi (plazmit)
plazmit_uzunluk = len(record.seq)
ax.plot([0, plazmit_uzunluk], [0, 0], 'k-', linewidth=2)

renkler = plt.cm.Set2.colors
i = 0

for f in record.features:
    if f.type != "CDS":
        continue
    
    gen = f.qualifiers.get("gene", ["?"])[0]
    bas = int(f.location.start)
    son = int(f.location.end)
    iplik = f.location.strand
    renk = renkler[i % len(renkler)]
    
    # Ok yönü ipliğe göre
    y = 0.4 if iplik == 1 else -0.4
    ok_yon = 1 if iplik == 1 else -1
    
    ax.annotate('', xy=(son if iplik == 1 else bas, y),
                xytext=(bas if iplik == 1 else son, y),
                arrowprops=dict(arrowstyle='->', color=renk, lw=8,
                               mutation_scale=15))
    
    ax.text((bas + son) / 2, y + (0.25 if iplik == 1 else -0.25),
            gen, ha='center', va='center', fontsize=9, fontweight='bold')
    
    i += 1

ax.set_xlim(-200, plazmit_uzunluk + 200)
ax.set_ylim(-1.2, 1.2)
ax.set_xlabel('Pozisyon (bp)')
ax.set_title(f'pPCP1 Plazmiti Haritası — {record.annotations.get("organism", "")} ({plazmit_uzunluk} bp)')
ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

# İplik açıklaması
ax.text(plazmit_uzunluk + 100, 0.4, '+ iplik', va='center', fontsize=8, color='gray')
ax.text(plazmit_uzunluk + 100, -0.4, '- iplik', va='center', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('plazmit_haritasi.png', dpi=150)
plt.show()
print("Harita kaydedildi: plazmit_haritasi.png")

---

## Uygulama 5 Tamamlandı!

**Öğrendiğin kavramlar:**
- GenBank formatı ve FASTA arasındaki fark
- `SeqRecord` nesnesinin yapısı (seq, id, annotations, features)
- Feature'lardan bilgi çıkarma (type, location, qualifiers)
- `feature.extract()` ile diziyi iplik-doğru kesme
- Bakteriyel genetik kod (tablo 11) ve alternatif başlangıç kodonları
- `SeqIO.write()` ile FASTA dosyası oluşturma
- `SeqIO.to_dict()` ile sözlük tabanlı erişim
- Matplotlib ile basit gen haritası çizme

---

## Tüm Uygulamalar Tamamlandı!

5 uygulama boyunca Hafta 1–3 konularını gerçek verilerle uyguladın:
1. Dizgi işlemleri, baz sayımı, GC oranı, motif
2. Transkripsiyon, translasyon, genetik kod tabloları
3. MutableSeq, nokta mutasyonu, hastalık analizi
4. PCR primer tasarımı, Tm hesaplama, GC profili
5. GenBank formatı, feature okuma, protein çıkarma